# Statistics — Advanced Reference for Data Scientists
> **Level:** Advanced | **Goal:** Practical statistical reasoning with Python

## Table of Contents
1. [Probability Distributions](#distributions)
2. [Descriptive Statistics & EDA](#descriptive)
3. [Hypothesis Testing](#hypothesis)
4. [Confidence Intervals & Effect Sizes](#ci)
5. [A/B Testing & Power Analysis](#ab)
6. [Bayesian Inference](#bayesian)
7. [Regression Analysis](#regression)
8. [Resampling Methods (Bootstrap & Permutation)](#resampling)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import (
    norm, t as t_dist, chi2, f as f_dist,
    ttest_ind, ttest_1samp, ttest_rel,
    mannwhitneyu, kruskal, chi2_contingency,
    shapiro, levene, pearsonr, spearmanr
)
import warnings; warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)
plt.rcParams.update({'figure.figsize': (10, 4), 'font.size': 11})
print("Setup complete")

---
## 1 · Probability Distributions <a id='distributions'></a>

| Distribution | Use case | Key params |
|---|---|---|
| Normal | Continuous measurements, CLT | μ, σ |
| Log-Normal | Revenue, prices, durations | μ, σ |
| Poisson | Event counts per interval | λ |
| Binomial | Success counts | n, p |
| Exponential | Time between events | λ |
| Beta | Probabilities, conversion rates | α, β |
| Gamma | Waiting times, aggregate amounts | k, θ |
| Uniform | Random sampling baseline | a, b |

In [ ]:
# ── SciPy distribution API ─────────────────────────────────────
# All distributions share: .pdf .cdf .ppf .rvs .stats .fit

dist = norm(loc=170, scale=10)  # height distribution

print(f"P(height < 180): {dist.cdf(180):.4f}")
print(f"P(160 < height < 180): {dist.cdf(180) - dist.cdf(160):.4f}")
print(f"90th percentile: {dist.ppf(0.90):.2f} cm")
print(f"Mean, Variance: {dist.stats()}")

# Fit a distribution to data
data = rng.lognormal(mean=5, sigma=0.8, size=1000)
mu_fit, sigma_fit = stats.lognorm.fit(data, floc=0)[2], stats.lognorm.fit(data, floc=0)[0]
print(f"\nFitted log-normal: mu={np.log(mu_fit):.2f}, sigma={sigma_fit:.2f}")

In [ ]:
# ── Central Limit Theorem visualization ────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
sample_sizes = [1, 5, 30, 100]
population = stats.expon(scale=2)  # skewed population

for ax, n in zip(axes, sample_sizes):
    sample_means = [population.rvs(n).mean() for _ in range(2000)]
    ax.hist(sample_means, bins=40, density=True, color='steelblue', alpha=0.7)
    ax.set_title(f'n={n}')
    ax.set_xlabel('Sample mean')

plt.suptitle('Central Limit Theorem: Exponential → Normal as n grows', y=1.02)
plt.tight_layout()
plt.show()

---
## 2 · Descriptive Statistics & EDA <a id='descriptive'></a>

In [ ]:
# ── Rich describe() extension ──────────────────────────────────
n = 1000
df = pd.DataFrame({
    'revenue':    rng.lognormal(5, 0.8, n),
    'sessions':   rng.poisson(8, n),
    'converted':  rng.binomial(1, 0.12, n),
    'channel':    rng.choice(['organic','paid','email','social'], n, p=[0.4,0.3,0.2,0.1])
})

def extended_describe(series):
    d = series.describe()
    d['skewness'] = series.skew()
    d['kurtosis'] = series.kurtosis()
    d['cv']       = series.std() / series.mean()  # coefficient of variation
    d['p95']      = series.quantile(0.95)
    d['p99']      = series.quantile(0.99)
    d['iqr']      = series.quantile(0.75) - series.quantile(0.25)
    return d.round(3)

print(extended_describe(df['revenue']))

In [ ]:
# ── Outlier detection methods ──────────────────────────────────
rev = df['revenue'].values

# Method 1: IQR fence
q1, q3 = np.percentile(rev, [25, 75])
iqr = q3 - q1
iqr_outliers = (rev < q1 - 1.5*iqr) | (rev > q3 + 1.5*iqr)

# Method 2: Z-score
z_scores = np.abs(stats.zscore(rev))
z_outliers = z_scores > 3

# Method 3: Modified Z-score (robust to outliers themselves)
median = np.median(rev)
mad = np.median(np.abs(rev - median))
mod_z = 0.6745 * np.abs(rev - median) / mad
modz_outliers = mod_z > 3.5

print(f"IQR method:       {iqr_outliers.sum()} outliers ({iqr_outliers.mean():.1%})")
print(f"Z-score method:   {z_outliers.sum()} outliers ({z_outliers.mean():.1%})")
print(f"Modified Z-score: {modz_outliers.sum()} outliers ({modz_outliers.mean():.1%})")

---
## 3 · Hypothesis Testing <a id='hypothesis'></a>

### Decision framework
```
1. State H₀ (null) and H₁ (alternative)
2. Choose significance level α (typically 0.05)
3. Check assumptions (normality, equal variance, independence)
4. Select test
5. Compute p-value
6. Decide: reject H₀ if p < α
7. Report effect size (not just p-value!)
```

| Test | When to use |
|---|---|
| One-sample t-test | Compare sample mean to known value |
| Two-sample t-test | Compare means of two independent groups |
| Paired t-test | Before/after on same subjects |
| Mann-Whitney U | Non-parametric two-group comparison |
| ANOVA / Kruskal-Wallis | 3+ groups |
| Chi-square | Categorical independence |

In [ ]:
# ── Assumption checks before t-test ───────────────────────────
control = rng.normal(100, 15, 150)
treatment = rng.normal(108, 15, 120)

# 1. Normality — Shapiro-Wilk (use on samples < 5000)
stat_c, p_norm_c = shapiro(control[:50])   # Shapiro works best on ≤ 50
stat_t, p_norm_t = shapiro(treatment[:50])
print(f"Normality — Control: p={p_norm_c:.4f}, Treatment: p={p_norm_t:.4f}")

# 2. Equal variances — Levene's test
stat_l, p_levene = levene(control, treatment)
print(f"Equal variances — Levene p={p_levene:.4f}")

# 3. Run appropriate test
equal_var = p_levene > 0.05
stat, p_val = ttest_ind(control, treatment, equal_var=equal_var)

# 4. Effect size — Cohen's d
pooled_std = np.sqrt((control.std()**2 + treatment.std()**2) / 2)
cohens_d = (treatment.mean() - control.mean()) / pooled_std

print(f"\nTwo-sample t-test (Welch={not equal_var}):")
print(f"  t={stat:.3f}, p={p_val:.4f}")
print(f"  Effect size (Cohen's d) = {cohens_d:.3f}")
print(f"  Interpretation: {'small' if abs(cohens_d) < 0.5 else 'medium' if abs(cohens_d) < 0.8 else 'large'}")

In [ ]:
# ── Chi-square test of independence ────────────────────────────
# Does conversion rate differ by channel?
contingency = pd.crosstab(df['channel'], df['converted'])
print(contingency)

chi2_stat, p_chi, dof, expected = chi2_contingency(contingency)
print(f"\nχ²={chi2_stat:.3f}, df={dof}, p={p_chi:.4f}")

# Cramér's V (effect size for chi-square)
n_total = contingency.sum().sum()
cramers_v = np.sqrt(chi2_stat / (n_total * (min(contingency.shape) - 1)))
print(f"Cramér's V = {cramers_v:.3f}  (0=no assoc, 1=perfect assoc)")

---
## 4 · Confidence Intervals & Effect Sizes <a id='ci'></a>

In [ ]:
# ── CI for mean (t-distribution) ──────────────────────────────
def ci_mean(data, confidence=0.95):
    n = len(data)
    se = stats.sem(data)
    h = se * t_dist.ppf((1 + confidence) / 2, df=n-1)
    return data.mean() - h, data.mean() + h

lo, hi = ci_mean(treatment)
print(f"Treatment mean 95% CI: [{lo:.2f}, {hi:.2f}]")

# ── CI for proportion ─────────────────────────────────────────
def ci_proportion(successes, n, confidence=0.95):
    """Wilson score interval — more accurate than normal approx."""
    p_hat = successes / n
    z = norm.ppf((1 + confidence) / 2)
    denominator = 1 + z**2 / n
    center = (p_hat + z**2 / (2*n)) / denominator
    half = z * np.sqrt(p_hat*(1-p_hat)/n + z**2/(4*n**2)) / denominator
    return center - half, center + half

conversions = df['converted'].sum()
lo_p, hi_p = ci_proportion(conversions, len(df))
print(f"Conversion rate 95% CI: [{lo_p:.3f}, {hi_p:.3f}]")

---
## 5 · A/B Testing & Power Analysis <a id='ab'></a>

In [ ]:
from scipy.stats import norm

def min_sample_size(
    baseline_rate: float,
    mde: float,           # minimum detectable effect (relative)
    alpha: float = 0.05,
    power: float = 0.80
) -> int:
    """Sample size per variant for a two-proportion z-test."""
    p1 = baseline_rate
    p2 = baseline_rate * (1 + mde)
    z_alpha = norm.ppf(1 - alpha / 2)  # two-tailed
    z_beta  = norm.ppf(power)
    pooled_p = (p1 + p2) / 2
    n = ((z_alpha * np.sqrt(2 * pooled_p * (1 - pooled_p)) +
          z_beta  * np.sqrt(p1*(1-p1) + p2*(1-p2))) / (p2 - p1)) ** 2
    return int(np.ceil(n))

# How many users per variant for a 10% lift on 5% baseline?
n = min_sample_size(baseline_rate=0.05, mde=0.10)
print(f"Required sample size per variant: {n:,}")

# Sensitivity analysis
print("\nSample size vs MDE:")
for mde in [0.05, 0.10, 0.15, 0.20, 0.30]:
    print(f"  MDE={mde:.0%}: {min_sample_size(0.05, mde):>10,} per variant")

In [ ]:
# ── Full A/B test analysis ─────────────────────────────────────
np.random.seed(0)
n_a, n_b = 5000, 5000
conv_a = rng.binomial(1, 0.05, n_a)
conv_b = rng.binomial(1, 0.055, n_b)  # 10% lift

# Observed
p_a = conv_a.mean()
p_b = conv_b.mean()
relative_lift = (p_b - p_a) / p_a

# Two-proportion z-test
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

stat, p_val = proportions_ztest([conv_b.sum(), conv_a.sum()], [n_b, n_a])
ci_a = proportion_confint(conv_a.sum(), n_a, method='wilson')
ci_b = proportion_confint(conv_b.sum(), n_b, method='wilson')

print(f"Control   CR: {p_a:.3%}  95% CI [{ci_a[0]:.3%}, {ci_a[1]:.3%}]")
print(f"Treatment CR: {p_b:.3%}  95% CI [{ci_b[0]:.3%}, {ci_b[1]:.3%}]")
print(f"Relative lift: {relative_lift:.1%}")
print(f"z={stat:.3f}, p={p_val:.4f}")
print(f"Decision: {'Reject H₀ — significant lift' if p_val < 0.05 else 'Fail to reject H₀'}")

---
## 6 · Bayesian Inference <a id='bayesian'></a>

In [ ]:
from scipy.stats import beta as beta_dist

# ── Beta-Binomial model for conversion rates ───────────────────
# Prior: Beta(α₀, β₀) — uninformative: α=β=1 (uniform)
alpha_prior, beta_prior = 1, 1

# Data
successes_a, trials_a = conv_a.sum(), n_a
successes_b, trials_b = conv_b.sum(), n_b

# Posterior: Beta(α₀ + successes, β₀ + failures)
post_a = beta_dist(alpha_prior + successes_a, beta_prior + trials_a - successes_a)
post_b = beta_dist(alpha_prior + successes_b, beta_prior + trials_b - successes_b)

# P(B > A) via Monte Carlo
n_samples = 100_000
samples_a = post_a.rvs(n_samples, random_state=42)
samples_b = post_b.rvs(n_samples, random_state=1)
prob_b_better = (samples_b > samples_a).mean()

# 95% Credible Intervals
ci_a_bayes = post_a.ppf([0.025, 0.975])
ci_b_bayes = post_b.ppf([0.025, 0.975])

print(f"P(B > A) = {prob_b_better:.3%}")
print(f"A posterior 95% CI: {ci_a_bayes}")
print(f"B posterior 95% CI: {ci_b_bayes}")

# Expected uplift
uplift_samples = (samples_b - samples_a) / samples_a
print(f"Expected relative uplift: {uplift_samples.mean():.2%}  (95% CI: {np.percentile(uplift_samples,[2.5,97.5])})") 

---
## 7 · Regression Analysis <a id='regression'></a>

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

# ── OLS regression with full diagnostics ──────────────────────
np.random.seed(42)
n_obs = 300
X_raw = pd.DataFrame({
    'age':        rng.integers(22, 65, n_obs),
    'experience': rng.integers(0, 20, n_obs),
    'education':  rng.choice([12, 14, 16, 18, 20], n_obs),
})
noise = rng.normal(0, 8000, n_obs)
salary = 30000 + 500*X_raw['age'] + 2000*X_raw['experience'] + 1500*X_raw['education'] + noise

X = sm.add_constant(X_raw)
model = sm.OLS(salary, X).fit()
print(model.summary())

In [ ]:
# ── Residual diagnostics ───────────────────────────────────────
residuals = model.resid
fitted = model.fittedvalues

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.4, s=15)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set(title='Residuals vs Fitted', xlabel='Fitted', ylabel='Residuals')

# 2. Q-Q plot
sm.qqplot(residuals, line='s', ax=axes[1])
axes[1].set_title('Q-Q Plot (Normality)')

# 3. Scale-Location (homoscedasticity)
axes[2].scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.4, s=15)
axes[2].set(title='Scale-Location', xlabel='Fitted', ylabel='√|Residuals|')

plt.tight_layout()
plt.show()

# Breusch-Pagan test for heteroscedasticity
bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X)
print(f"Breusch-Pagan test: stat={bp_stat:.3f}, p={bp_p:.4f}")
print(f"Homoscedasticity: {'OK' if bp_p > 0.05 else 'VIOLATION'}")

---
## 8 · Resampling Methods <a id='resampling'></a>

In [ ]:
# ── Bootstrap CI for any statistic ────────────────────────────
def bootstrap_ci(data, statistic=np.mean, n_boot=10000, ci=0.95, seed=42):
    """Non-parametric bootstrap confidence interval."""
    rng_b = np.random.default_rng(seed)
    boot_stats = [
        statistic(rng_b.choice(data, size=len(data), replace=True))
        for _ in range(n_boot)
    ]
    alpha = 1 - ci
    lo, hi = np.percentile(boot_stats, [alpha/2*100, (1-alpha/2)*100])
    return statistic(data), lo, hi, boot_stats

observed_rev = df['revenue'].values

mean_obs, lo, hi, _ = bootstrap_ci(observed_rev, np.mean)
med_obs,  lo_m, hi_m, _ = bootstrap_ci(observed_rev, np.median)

print(f"Mean   = {mean_obs:.2f}  95% Bootstrap CI: [{lo:.2f}, {hi:.2f}]")
print(f"Median = {med_obs:.2f}  95% Bootstrap CI: [{lo_m:.2f}, {hi_m:.2f}]")

In [ ]:
# ── Permutation test (model-free hypothesis test) ──────────────
def permutation_test(group_a, group_b, n_perm=10000, seed=42):
    """Test if two groups have different means using permutation."""
    rng_p = np.random.default_rng(seed)
    observed_diff = group_b.mean() - group_a.mean()
    combined = np.concatenate([group_a, group_b])
    n_a = len(group_a)

    perm_diffs = np.array([
        (perm := rng_p.permutation(combined))[:n_a].mean() - perm[n_a:].mean() * -1
        for _ in range(n_perm)
    ])
    p_val = (np.abs(perm_diffs) >= np.abs(observed_diff)).mean()
    return observed_diff, p_val

obs_diff, perm_p = permutation_test(control, treatment)
print(f"Observed difference: {obs_diff:.3f}")
print(f"Permutation p-value: {perm_p:.4f}")

---
## 📝 Quiz Notes — Hypothesis Testing: Tail Types <a id='tail-tests'></a>

### Question
> Ben is given three graphs, each with a shaded (red) area labelled A, B, or C.  
> Which statements correctly describe each graph?

### ✅ Correct answer — Option 1

| Graph | Shaded area | Test type | $H_a$ |
|---|---|---|---|
| **1 — A** | Right tail | Right-tailed (one-sided) | $H_a: \mu > value$ |
| **2 — B** | Left tail | Left-tailed (one-sided) | $H_a: \mu < value$ |
| **3 — C** | Both tails | Two-tailed | $H_a: \mu \neq value$ |

### How to read the graphs

The **shaded area represents the rejection region** — where, if the test statistic falls, you reject $H_0$.

```
Right-tail (Graph 1):          Left-tail (Graph 2):         Two-tail (Graph 3):

     ___                              ___                        ___
    /   \                            /   \                      /   \
   /     \     ░░░░             ░░░░/     \                 ░░░/     \░░░
──/───────\─────────           ──────────/──────           ──/─────────\──
           0    →                      ←  0                    ←  0  →
  H_a: μ > value               H_a: μ < value             H_a: μ ≠ value
```

### The three hypothesis test types

| Test | When to use | $H_0$ | $H_a$ | $\alpha$ split |
|---|---|---|---|---|
| **Right-tailed** | Testing if mean is **greater than** a value | $\mu \leq value$ | $\mu > value$ | All $\alpha$ in right tail |
| **Left-tailed** | Testing if mean is **less than** a value | $\mu \geq value$ | $\mu < value$ | All $\alpha$ in left tail |
| **Two-tailed** | Testing if mean is **different from** a value | $\mu = value$ | $\mu \neq value$ | $\alpha/2$ in each tail |

### P-value interpretation by test type

```
Right-tail:   p-value = P(Z ≥ z_observed)
Left-tail:    p-value = P(Z ≤ z_observed)
Two-tail:     p-value = 2 × P(Z ≥ |z_observed|)
```

> **Rule:** If $p\text{-value} < \alpha$ → reject $H_0$, evidence supports $H_a$

In [ ]:
# ── Tail tests visualized ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.linspace(-4, 4, 400)
pdf = stats.norm.pdf(x)
alpha = 0.05

titles  = ['Graph 1 — Right-tail\n$H_a: \\mu > value$',
           'Graph 2 — Left-tail\n$H_a: \\mu < value$',
           'Graph 3 — Two-tail\n$H_a: \\mu \\neq value$']
colors  = ['crimson', 'crimson', 'crimson']

for ax, title in zip(axes, titles):
    ax.plot(x, pdf, 'k-', lw=2)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel('Test statistic')
    ax.set_ylabel('Density')
    ax.spines[['top','right']].set_visible(False)
    ax.set_xlim(-4, 4)
    ax.set_xticks([0])
    ax.set_xticklabels(['0 (μ₀)'])

# Right-tail
z_right = stats.norm.ppf(1 - alpha)
axes[0].fill_between(x, pdf, where=(x >= z_right), color='crimson', alpha=0.7, label=f'α={alpha}')
axes[0].axvline(z_right, color='navy', ls='--', lw=1.5, label=f'z_crit={z_right:.2f}')
axes[0].annotate('A', xy=(z_right+0.3, 0.02), fontsize=13, color='crimson', fontweight='bold')
axes[0].legend(fontsize=8)

# Left-tail
z_left = stats.norm.ppf(alpha)
axes[1].fill_between(x, pdf, where=(x <= z_left), color='crimson', alpha=0.7, label=f'α={alpha}')
axes[1].axvline(z_left, color='navy', ls='--', lw=1.5, label=f'z_crit={z_left:.2f}')
axes[1].annotate('B', xy=(z_left-0.8, 0.02), fontsize=13, color='crimson', fontweight='bold')
axes[1].legend(fontsize=8)

# Two-tail
z_two = stats.norm.ppf(1 - alpha/2)
axes[2].fill_between(x, pdf, where=(x >= z_two),  color='crimson', alpha=0.7, label=f'α/2={alpha/2}')
axes[2].fill_between(x, pdf, where=(x <= -z_two), color='crimson', alpha=0.7)
axes[2].axvline( z_two, color='navy', ls='--', lw=1.5)
axes[2].axvline(-z_two, color='navy', ls='--', lw=1.5, label=f'z_crit=±{z_two:.2f}')
axes[2].annotate('C', xy=(z_two+0.15, 0.02), fontsize=13, color='crimson', fontweight='bold')
axes[2].annotate('C', xy=(-z_two-0.6, 0.02), fontsize=13, color='crimson', fontweight='bold')
axes[2].legend(fontsize=8)

plt.suptitle('Hypothesis Test Types — Rejection Regions (α = 0.05)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── P-value calculation example ───────────────────────────────
print("── P-value examples for z = 1.8 ──")
z = 1.8
print(f"  Right-tail:  p = P(Z ≥ {z}) = {1 - stats.norm.cdf(z):.4f}")
print(f"  Left-tail:   p = P(Z ≤ {z}) = {stats.norm.cdf(z):.4f}")
print(f"  Two-tail:    p = 2×P(Z ≥ |{z}|) = {2*(1 - stats.norm.cdf(abs(z))):.4f}")
print(f"\n  α = 0.05 → reject H₀ if p < 0.05")
print(f"  Right-tail: {1 - stats.norm.cdf(z):.4f} < 0.05? {1 - stats.norm.cdf(z) < 0.05}")